# NHANES III — Full Batch Processing on Google Colab

Downloads every bilateral knee X-ray from the CDC FTP, processes each one, and saves 224x224 PNGs permanently to **your Google Drive**.

---

## Before you run

### Enable Background Execution (so closing your browser does NOT stop the job)
1. In Colab, go to **Runtime** in the top menu
2. Click **"Change runtime type"** → make sure you are on **Colab Pro or Pro+**
3. Back in **Runtime**, click **"Run all"** — a banner will appear at the top offering **"Run in background"** → click it
4. You can now **close the browser tab and shut down your PC** — the job keeps running on Google's servers

> **Free Colab:** Background execution is not available. Keep the browser tab open. The runtime disconnects after ~90 min of browser inactivity.

---

## How to run
1. **File → Open notebook** in Colab → upload this file
2. Click **Runtime → Run all**
3. A Google Drive authorisation popup appears in Cell 1 — approve it
4. Everything else runs automatically

---

## What it does (one image at a time)
```
Download TIFF (~38 MB) → normalise → split L/R → CLAHE → resize 224x224 → save PNGs to Drive → delete TIFF → next
```
At no point is more than ~38 MB of raw data held locally.  
Checkpoint is written after every image — safe to interrupt and resume at any time.

## Output filenames
```
NHANES3_SEQN10018_R_norm-pct1-99_clahe-2.0_224px.png
NHANES3_SEQN10018_L_norm-pct1-99_clahe-2.0_224px.png
```

In [ ]:


from google.colab import drive
drive.mount("/content/drive")
print("Google Drive mounted.")

Mounted at /content/drive
Google Drive mounted.


In [ ]:

import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "tifffile", "opencv-python-headless", "Pillow", "numpy", "tqdm"
])
print("Packages ready.")

Packages ready.


In [ ]:

import datetime
import json
import re
import urllib.error
import urllib.request
from pathlib import Path

import cv2
import numpy as np
import tifffile
from PIL import Image
from tqdm.notebook import tqdm

print("Imports OK.")

Imports OK.


---
## Configuration
The only thing you may want to change is `OUT_DIR` if you prefer a different folder name in your Drive.

In [ ]:


DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT    = DRIVE_ROOT / "Master Thesis"

OUT_DIR  = PROJECT / "nhanes3" / "processed"

TIFF_TMP = Path("/content/tiff_tmp")

FTP_BASE        = "https://ftp.cdc.gov/pub/NHANES/XRays/Nhanes3/"
CHECKPOINT_FILE = OUT_DIR / "checkpoint.json"
ERROR_LOG_FILE  = OUT_DIR / "errors.log"
PNG_SUFFIX      = "norm-pct1-99_clahe-2.0_224px"
DOWNLOAD_TIMEOUT = 120

OUT_DIR.mkdir(parents=True, exist_ok=True)
TIFF_TMP.mkdir(parents=True, exist_ok=True)

print(f"Output (Drive) : {OUT_DIR}")
print(f"Temp TIFF      : {TIFF_TMP}")
print(f"Checkpoint     : {CHECKPOINT_FILE}")


Output (Drive) : /content/drive/MyDrive/nhanes3/processed
Temp TIFF      : /content/tiff_tmp
Checkpoint     : /content/drive/MyDrive/nhanes3/processed/checkpoint.json


In [ ]:


def normalise_film(arr: np.ndarray) -> np.ndarray:
    """Stretch 1st-99th percentile of a 16-bit array to uint8 [0, 255]."""
    arr = arr.astype(np.float32)
    lo, hi = float(np.percentile(arr, 1)), float(np.percentile(arr, 99))
    if hi - lo < 1e-6:
        lo, hi = float(arr.min()), float(arr.max())
    if hi - lo < 1e-6:
        return np.zeros_like(arr, dtype=np.uint8)
    return np.clip((arr - lo) / (hi - lo) * 255.0, 0, 255).astype(np.uint8)

def process_and_save(tiff_path: Path, seqn: str):
    """Full pipeline: one TIFF → two 224x224 PNGs saved to Drive."""
    raw = tifffile.imread(str(tiff_path))
    if raw.ndim == 3:
        raw = raw[..., 0]

    norm       = normalise_film(raw)
    mid        = norm.shape[1] // 2
    crop_right = norm[:, :mid]
    crop_left  = np.fliplr(norm[:, mid:])

    clahe      = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    n          = int(seqn)
    r_path     = OUT_DIR / f"NHANES3_SEQN{n:05d}_R_{PNG_SUFFIX}.png"
    l_path     = OUT_DIR / f"NHANES3_SEQN{n:05d}_L_{PNG_SUFFIX}.png"

    Image.fromarray(clahe.apply(crop_right), "L").resize((224, 224), Image.LANCZOS).save(str(r_path))
    Image.fromarray(clahe.apply(crop_left),  "L").resize((224, 224), Image.LANCZOS).save(str(l_path))

def load_checkpoint() -> set:
    if CHECKPOINT_FILE.exists():
        try:
            return set(json.loads(CHECKPOINT_FILE.read_text()))
        except (json.JSONDecodeError, OSError):
            pass
    return set()

def save_checkpoint(done: set) -> None:
    CHECKPOINT_FILE.write_text(json.dumps(sorted(done, key=int), indent=2))

print("Functions defined.")

Functions defined.


In [ ]:

print("Fetching FTP listing ...")
try:
    req  = urllib.request.Request(FTP_BASE, headers={"User-Agent": "Mozilla/5.0"})
    html = urllib.request.urlopen(req, timeout=60).read().decode("utf-8", errors="ignore")
except urllib.error.URLError as e:
    raise RuntimeError(f"Cannot reach CDC FTP: {e}")

all_seqns = sorted(
    set(re.findall(r'>(\d+)K1\.tiff<', html, re.IGNORECASE)),
    key=int,
)

if len(all_seqns) < 100:

    print("WARNING: fewer than 100 SEQNs found — raw HTML snippet:")
    print(html[:3000])
    raise RuntimeError(f"Only {len(all_seqns)} SEQNs found — something is wrong with the FTP response.")

print(f"Found {len(all_seqns)} K1 TIFFs on FTP")
print(f"SEQN range : {all_seqns[0]} – {all_seqns[-1]}")

Fetching FTP listing ...
Found 4905 K1 TIFFs on FTP
SEQN range : 49 – 53616


In [ ]:

already_done = load_checkpoint()
todo         = [s for s in all_seqns if s not in already_done]

print(f"Total on FTP  : {len(all_seqns)}")
print(f"Already done  : {len(already_done)}")
print(f"Remaining     : {len(todo)}")
if already_done:
    s = sorted(already_done, key=int)
    print(f"Done range    : {s[0]} – {s[-1]}")

Total on FTP  : 4905
Already done  : 12
Remaining     : 4893
Done range    : 49 – 139


---
## Main Loop

This is the long-running cell (~9 hours for all ~3300 images).

- **To run in background:** before clicking Run All, go to **Runtime → Run all** and accept the "Run in background" offer (Colab Pro/Pro+)
- **To interrupt:** Runtime → Interrupt execution — progress is always saved
- **To resume:** just run this cell again — it picks up from the checkpoint automatically

In [ ]:


errors       = {}
skip_set     = already_done.copy()
tmp_file     = None
run_start    = datetime.datetime.now()

try:
    for seqn in tqdm(todo, desc="Processing", unit="img"):

        n      = int(seqn)
        r_path = OUT_DIR / f"NHANES3_SEQN{n:05d}_R_{PNG_SUFFIX}.png"
        l_path = OUT_DIR / f"NHANES3_SEQN{n:05d}_L_{PNG_SUFFIX}.png"

        if r_path.exists() and l_path.exists():
            skip_set.add(seqn)
            save_checkpoint(skip_set)
            continue

        url      = f"{FTP_BASE}{seqn}K1.tiff"
        tmp_file = TIFF_TMP / f"{seqn}K1.tiff"
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=DOWNLOAD_TIMEOUT) as resp,\
                 open(tmp_file, "wb") as fh:
                fh.write(resp.read())
        except Exception as e:
            errors[seqn] = f"download: {e}"
            if tmp_file and tmp_file.exists():
                tmp_file.unlink()
            tmp_file = None
            continue

        try:
            process_and_save(tmp_file, seqn)
        except Exception as e:
            errors[seqn] = f"process: {e}"
            tmp_file.unlink(missing_ok=True)
            tmp_file = None
            continue

        tmp_file.unlink(missing_ok=True)
        tmp_file = None

        skip_set.add(seqn)
        save_checkpoint(skip_set)

except KeyboardInterrupt:
    print("\nInterrupted. Progress saved — re-run this cell to resume.")
    if tmp_file and tmp_file.exists():
        tmp_file.unlink()

if errors:
    with open(ERROR_LOG_FILE, "a") as f:
        for s, msg in errors.items():
            f.write(f"{s}\t{msg}\n")

elapsed = datetime.datetime.now() - run_start
print(f"\nRun started     : {run_start.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Elapsed         : {str(elapsed).split('.')[0]}")
print(f"Done this run   : {len(skip_set) - len(already_done)}")
print(f"Total completed : {len(skip_set)} / {len(all_seqns)} SEQNs")
print(f"Errors          : {len(errors)}" + (f"  → see {ERROR_LOG_FILE}" if errors else "  (none)"))

Processing:   0%|          | 0/4893 [00:00<?, ?img/s]

/tmp/ipykernel_11647/3650025254.py:30: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(clahe.apply(crop_right), "L").resize((224, 224), Image.LANCZOS).save(str(r_path))
/tmp/ipykernel_11647/3650025254.py:31: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(clahe.apply(crop_left),  "L").resize((224, 224), Image.LANCZOS).save(str(l_path))



Run started     : 2026-04-20 16:13:33
Elapsed         : 3:33:48
Done this run   : 4893
Total completed : 4905 / 4905 SEQNs
Errors          : 0  (none)


In [ ]:

pngs    = sorted(OUT_DIR.glob("NHANES3_SEQN*_*_*.png"))
done_ck = load_checkpoint()

print(f"SEQNs in checkpoint : {len(done_ck)}")
print(f"Expected PNGs       : {len(done_ck) * 2}")
print(f"Actual PNGs on Drive: {len(pngs)}")

if len(pngs) < len(done_ck) * 2:
    present = {re.search(r"SEQN(\d+)", p.name).group(1) for p in pngs if re.search(r"SEQN(\d+)", p.name)}
    missing = done_ck - present
    print(f"\nMissing SEQNs ({len(missing)}): {sorted(missing, key=int)[:20]}")
    print("Re-run Cell 8 to fill in the gaps.")
else:
    size_mb = sum(p.stat().st_size for p in pngs) / 1e6
    print(f"\nAll {len(done_ck)} SEQNs complete.")
    print(f"Total size on Drive : {size_mb:.1f} MB")

if pngs:
    print(f"\nSample filenames:")
    for p in pngs[:4]:
        print(f"  {p.name}")

SEQNs in checkpoint : 4905
Expected PNGs       : 9810
Actual PNGs on Drive: 9810

All 4905 SEQNs complete.
Total size on Drive : 208.0 MB

Sample filenames:
  NHANES3_SEQN00049_L_norm-pct1-99_clahe-2.0_224px.png
  NHANES3_SEQN00049_R_norm-pct1-99_clahe-2.0_224px.png
  NHANES3_SEQN00063_L_norm-pct1-99_clahe-2.0_224px.png
  NHANES3_SEQN00063_R_norm-pct1-99_clahe-2.0_224px.png


---
## Inspect errors (optional)
Run the cell below if errors were reported in Cell 8.  
Failed SEQNs are **not** in the checkpoint, so re-running Cell 8 will retry them automatically.

In [ ]:

if ERROR_LOG_FILE.exists():
    lines = ERROR_LOG_FILE.read_text().strip().splitlines()
    print(f"{len(lines)} error(s):\n")
    for ln in lines:
        print(f"  {ln}")
else:
    print("No errors — all images processed successfully.")

No errors — all images processed successfully.
